In [1]:
import numpy as np
import pandas as pd
import xarray as xr

from scipy.stats import median_abs_deviation

In [2]:
regions = np.arange(1,20,1)
outpath = '/g/data/rd53/wy2165/disequilibrium/pygem_oggm/'
datapath = '/g/data/rd53/wy2165/disequilibrium/data/'

In [3]:
result = pd.read_csv(datapath + 'temp_ch_ipcc_ar6_isimip3b.csv', index_col=0)
result['AAR_steady_mean']               = np.nan
result['AAR_steady_std']                = np.nan
result['AAR_steady_median']             = np.nan
result['AAR_steady_MAD']                = np.nan
result['AAR_steady_area_weighted_mean'] = np.nan

result['AAR_mean_mean']               = np.nan
result['AAR_mean_std']                = np.nan
result['AAR_mean_median']             = np.nan
result['AAR_mean_MAD']                = np.nan
result['AAR_mean_area_weighted_mean'] = np.nan

result['disequilibrium_mean']               = np.nan
result['disequilibrium_std']                = np.nan
result['disequilibrium_median']             = np.nan
result['disequilibrium_MAD']                = np.nan
result['disequilibrium_area_weighted_mean'] = np.nan

result

,gcm,period_scenario,temp_ch_ipcc,AAR_steady_mean,AAR_steady_std,AAR_steady_median,AAR_steady_MAD,AAR_steady_area_weighted_mean,AAR_mean_mean,AAR_mean_std,AAR_mean_median,AAR_mean_MAD,AAR_mean_area_weighted_mean,disequilibrium_mean,disequilibrium_std,disequilibrium_median,disequilibrium_MAD,disequilibrium_area_weighted_mean
0,era5,2014-2023,1.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gfdl-esm4,1851-1870_hist,0.231409,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gfdl-esm4,1901-1920_hist,0.478289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gfdl-esm4,1951-1970_hist,0.392281,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gfdl-esm4,1995-2014_hist,0.901467,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,ukesm1-0-ll,2081-2100_ssp370,5.840495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,ukesm1-0-ll,2021-2040_ssp585,2.319733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,ukesm1-0-ll,2041-2060_ssp585,3.646968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,ukesm1-0-ll,2061-2080_ssp585,5.230544,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
ds_list = []

for region in regions:
    fn = outpath + f'PyGEM_glacier_stats_{region:02d}_median_k.nc'
    
    ds = xr.open_dataset(fn)
    
    ds = ds.assign_coords(rgi_id=ds['rgi_id'].astype(str))
    
    ds = ds.assign(
        region=('rgi_id', np.full(ds.sizes['rgi_id'], region, dtype=int))
    )
    
    ds_list.append(ds)

ds_all = xr.concat(
    ds_list,
    dim='rgi_id',
    data_vars='all',
    coords='minimal',
    compat='override',
    join='override'
)

ds_all = ds_all.sortby('rgi_id')

In [5]:
rgi_ids = ds_all['rgi_id'].values
unique_ids, counts = np.unique(rgi_ids, return_counts=True)

if np.any(counts > 1):
    duplicated = unique_ids[counts > 1]
    print(f'Warning: found duplicated rgi_id: {len(duplicated)}')
    print(duplicated[:20])

In [6]:
area_init = ds_all['rgi_area_km2']

for i in range(81):
    ########################################### AAR_steady
    AAR_steady = ds_all['AAR_steady'].values[:,i]
    # mean
    result.loc[i,'AAR_steady_mean'] = np.nanmean(AAR_steady)
    # std
    result.loc[i,'AAR_steady_std'] = np.nanstd(AAR_steady)
    # median
    result.loc[i,'AAR_steady_median'] = np.nanmedian(AAR_steady)
    # MAD: median absolute deviation from median
    result.loc[i,'AAR_steady_MAD'] = median_abs_deviation(AAR_steady, nan_policy='omit')
    # area-weighted mean
    mask = np.isfinite(AAR_steady) & np.isfinite(area_init) & (area_init > 0)
    aar_valid = AAR_steady[mask]
    area_valid = area_init[mask]
    result.loc[i,'AAR_steady_area_weighted_mean'] = np.sum(aar_valid * area_valid) / np.sum(area_valid)

    ########################################### AAR_mean
    AAR_mean = ds_all['AAR_mean'].values[:,i]
    # mean
    result.loc[i,'AAR_mean_mean'] = np.nanmean(AAR_mean)
    # std
    result.loc[i,'AAR_mean_std'] = np.nanstd(AAR_mean)
    # median
    result.loc[i,'AAR_mean_median'] = np.nanmedian(AAR_mean)
    # MAD: median absolute deviation from median
    result.loc[i,'AAR_mean_MAD'] = median_abs_deviation(AAR_mean, nan_policy='omit')
    # area-weighted mean
    mask = np.isfinite(AAR_mean) & np.isfinite(area_init) & (area_init > 0)
    aar_valid = AAR_mean[mask]
    area_valid = area_init[mask]
    result.loc[i,'AAR_mean_area_weighted_mean'] = np.sum(aar_valid * area_valid) / np.sum(area_valid)

    ########################################### disequilibrium
    disequilibrium = ds_all['disequilibrium'].values[:,i]
    # mean
    result.loc[i,'disequilibrium_mean'] = np.nanmean(disequilibrium)
    # std
    result.loc[i,'disequilibrium_std'] = np.nanstd(disequilibrium)
    # median
    result.loc[i,'disequilibrium_median'] = np.nanmedian(disequilibrium)
    # MAD: median absolute deviation from median
    result.loc[i,'disequilibrium_MAD'] = median_abs_deviation(disequilibrium, nan_policy='omit')
    # area-weighted mean
    mask = np.isfinite(disequilibrium) & np.isfinite(area_init) & (area_init > 0)
    disequilibrium_valid = disequilibrium[mask]
    area_valid = area_init[mask]
    result.loc[i,'disequilibrium_area_weighted_mean'] = np.sum(disequilibrium_valid * area_valid) / np.sum(area_valid)

In [7]:
result.to_csv(outpath + f'PyGEM_global_stats_median_k.csv')

In [8]:
ds_all.to_netcdf(
    outpath + f'PyGEM_global_glacier_stats_median_k.nc'
)